In [1]:
import torch 
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10


In [5]:
# Datasets & DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([  # help with multiple transformations at once  
    transforms.ToTensor(), # to convert the data into tensors amnd then scale them to (0,1)
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)) # Normalize the data to (-1,1)
])

trainset = CIFAR10(root = "./data", train = True, download = True, transform = transform)
testset = CIFAR10(root = "./data", train = False, download = True, transform = transform)

100%|████████████████████████████████████████| 170M/170M [00:02<00:00, 60.4MB/s]


In [6]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [9]:
trainLoader = DataLoader(trainset, batch_size = 64, shuffle = True)
testLoader = DataLoader(testset, batch_size = 64)

### Build the CNN

In [19]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32 ,kernel_size = 3, padding = 1), # input,output, kernel_size, padding 
            nn.ReLU(),
            nn.MaxPool2d(2,2), # Kernel_size = 2, stride = 2

            nn.Conv2d(32, 64, kernel_size = 3, padding = 1), # input,output, kernel_size, padding 
            nn.ReLU(),
            nn.MaxPool2d(2,2), # Kernel_size = 2, stride = 2

            nn.Conv2d(64, 128, kernel_size = 3, padding = 1), # input,output, kernel_size, padding 
            nn.ReLU(),
            nn.MaxPool2d(2,2) # Kernel_size = 2, stride = 2            
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flattening step 
        x = self.fc_layers(x)\

        return x

In [20]:
model = CNN()

In [21]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training CNN

In [32]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainLoader:
        optimizer.zero_grad() # setting gradients to zero after every loop

        output = model.forward(images) # FP
        loss = criterion(output, labels) # loss fnx
        loss.backward() # BP
        optimizer.step() # update parametrs 

        epoch_training_loss += loss.item()  # convert into a floating item as it is in tensor value

    print(f"epoch = {epoch +1} / {epochs} & loss = {epoch_training_loss/ len(trainLoader)}")   

epoch = 1 / 10 & loss = 0.8352001490800277
epoch = 2 / 10 & loss = 0.6857770777038296
epoch = 3 / 10 & loss = 0.5725515386103974
epoch = 4 / 10 & loss = 0.4751170246345003
epoch = 5 / 10 & loss = 0.386345588452066
epoch = 6 / 10 & loss = 0.3125748682833846
epoch = 7 / 10 & loss = 0.242973795708488
epoch = 8 / 10 & loss = 0.18797392640119928
epoch = 9 / 10 & loss = 0.148940259543107
epoch = 10 / 10 & loss = 0.12538439581346938


In [44]:
# Evaluate our CNN model using accuracy

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad(): # cuz we are doing no backward propogation
    for images,labels in testLoader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100} ")      

accuracy = 73.25 
